# Worker Utility

> Worker utility functions and the spatial indifference principle


In [ ]:
# | default_exp utility

In [ ]:
# | export
import numpy as np

## Why Do Wages and Rents Differ Across Cities?

Consider two empirical facts about London:

- London wages are ~30% higher than the UK average
- London rents are ~3x the UK average

**Question**: Why would anyone live in London? The wages don't seem to compensate for the rent!

The Rosen-Roback model answers this by recognizing that **locations differ in ways beyond just wages and rents**:

- **Amenities**: restaurants, culture, weather, scenery, family ties
- **Productivity**: some places make workers more productive (agglomeration)

### The Spatial Indifference Principle

In equilibrium, workers must be **indifferent** between locations. If one city were strictly better, everyone would move there. This is called **spatial equilibrium**.

$$V_1 = V_2 = V_3 = \ldots = \bar{V}$$

where $V_i$ is the utility a worker gets from living in city $i$.

### What Determines Utility?

Workers care about:

1. **Wages** ($w$): How much they earn
2. **Rents** ($r$): How much they pay for housing
3. **Amenities** ($a$): Everything else (weather, culture, etc.)

We'll use the CES (Constant Elasticity of Substitution) utility function:

$$V = \frac{w}{r^\beta} \cdot a$$

where:

- $\beta$ is the **housing expenditure share** (typically ~0.33, meaning workers spend 1/3 of income on housing)
- Higher wages increase utility
- Higher rents decrease utility (with strength determined by $\beta$)
- Higher amenities increase utility


## Utility Functions


In [ ]:
# | export
# | echo: true
def calculate_utility(wage: float, rent: float, amenity: float = 1.0, beta: float = 0.33) -> float:
    """Calculate worker utility using CES specification."""
    return (wage / (rent**beta)) * amenity


def utility_compensating_wage(rent_ratio: float, amenity_ratio: float = 1.0, beta: float = 0.33) -> float:
    """Calculate the wage ratio needed to maintain constant utility."""
    return (rent_ratio**beta) / amenity_ratio

Examples:


In [ ]:
# Calculate utility for a simple case
u = calculate_utility(wage=1000, rent=500, beta=0.33)
assert u > 0

# If London rent is 3x higher, wage must be ~1.44x higher to compensate
wage_ratio = utility_compensating_wage(rent_ratio=3.0, beta=0.33)
assert wage_ratio > 1.4 and wage_ratio < 1.5

## The Role of Beta (Housing Expenditure Share)

The parameter $\beta$ controls how much workers care about rents relative to wages.

- If $\beta = 0$: Workers don't care about rents at all (unrealistic!)
- If $\beta = 0.33$: Workers spend 1/3 of income on housing (realistic)
- If $\beta = 0.5$: Workers spend half their income on housing (high cost area)

**Implication**: If workers spend more on housing (higher $\beta$), they are MORE sensitive to rent differences. This makes them less willing to move to high-rent cities.


## Tests


In [ ]:
# | hide
# Test basic utility calculation
u1 = calculate_utility(wage=1000, rent=500, beta=0.33)
assert u1 > 0
assert np.isclose(u1, 1000 / (500**0.33), rtol=0.01)

# Test that higher wages increase utility
u_low_wage = calculate_utility(wage=1000, rent=500)
u_high_wage = calculate_utility(wage=2000, rent=500)
assert u_high_wage > u_low_wage

# Test that higher rents decrease utility
u_low_rent = calculate_utility(wage=1000, rent=500)
u_high_rent = calculate_utility(wage=1000, rent=1000)
assert u_low_rent > u_high_rent

# Test amenity effects
u_no_amenity = calculate_utility(wage=1000, rent=500, amenity=1.0)
u_with_amenity = calculate_utility(wage=1000, rent=500, amenity=1.5)
assert u_with_amenity > u_no_amenity
assert np.isclose(u_with_amenity / u_no_amenity, 1.5)

# Test compensating wage
# If rent is 3x higher, wage must be ~1.44x higher (with beta=0.33)
wage_ratio = utility_compensating_wage(rent_ratio=3.0, beta=0.33)
assert np.isclose(wage_ratio, 3.0**0.33, rtol=0.01)

## Example: London vs Manchester

Let's use the utility function to understand the London-Manchester wage and rent differences.


In [ ]:
# Example cities (stylized facts)
london_wage = 45_000
london_rent = 20_000
london_amenity = 1.2  # 20% amenity premium

manchester_wage = 35_000
manchester_rent = 8_000
manchester_amenity = 1.0  # Baseline

beta = 0.33

# Calculate utilities
london_utility = calculate_utility(london_wage, london_rent, london_amenity, beta)
manchester_utility = calculate_utility(manchester_wage, manchester_rent, manchester_amenity, beta)

print("City Comparison:")
print("=" * 60)
print(f"{'City':<15} {'Wage':>12} {'Rent':>12} {'Amenity':>10} {'Utility':>10}")
print("-" * 60)
print(f"{'London':<15} £{london_wage:>10,} £{london_rent:>10,} {london_amenity:>10.2f} {london_utility:>10.2f}")
print(
    f"{'Manchester':<15} £{manchester_wage:>10,} £{manchester_rent:>10,} {manchester_amenity:>10.2f} {manchester_utility:>10.2f}"
)
print("-" * 60)

# Check if they're in equilibrium
utility_diff_pct = abs(london_utility - manchester_utility) / manchester_utility * 100
print(f"\nUtility difference: {utility_diff_pct:.1f}%")

if utility_diff_pct < 5:
    print("✓ Cities are approximately in spatial equilibrium")
else:
    print("✗ Not in equilibrium - workers would migrate")

# Calculate required wage for exact equilibrium
rent_ratio = london_rent / manchester_rent
amenity_ratio = london_amenity / manchester_amenity
required_wage_ratio = utility_compensating_wage(rent_ratio, amenity_ratio, beta)
required_london_wage = manchester_wage * required_wage_ratio

print("\nFor exact equilibrium:")
print(f"  Current London wage: £{london_wage:,}")
print(f"  Required London wage: £{required_london_wage:,.0f}")
print(f"  Difference: £{london_wage - required_london_wage:,.0f}")

City Comparison:
City                    Wage         Rent    Amenity    Utility
------------------------------------------------------------
London          £    45,000 £    20,000       1.20    2056.15
Manchester      £    35,000 £     8,000       1.00    1803.22
------------------------------------------------------------

Utility difference: 14.0%
✗ Not in equilibrium - workers would migrate

For exact equilibrium:
  Current London wage: £45,000
  Required London wage: £39,465
  Difference: £5,535


## Sensitivity to Beta

How does the housing expenditure share affect spatial equilibrium?


In [6]:
# Vary beta from 0.2 to 0.5
betas = [0.20, 0.25, 0.30, 0.33, 0.40, 0.50]
rent_ratio = 3.0  # London rent is 3x Manchester

print("How housing expenditure share affects wage compensation:")
print("=" * 60)
print(f"{'Beta':>8} {'Housing Share':>15} {'Required Wage Ratio':>20}")
print("-" * 60)

for beta in betas:
    wage_ratio = utility_compensating_wage(rent_ratio, beta=beta)
    print(f"{beta:>8.2f} {beta * 100:>14.0f}% {wage_ratio:>19.2f}x")

print("-" * 60)
print("\nInterpretation: Higher beta → workers care more about rents")
print("                 → need higher wage compensation to move to expensive city")

How housing expenditure share affects wage compensation:
    Beta   Housing Share  Required Wage Ratio
------------------------------------------------------------
    0.20             20%                1.25x
    0.25             25%                1.32x
    0.30             30%                1.39x
    0.33             33%                1.44x
    0.40             40%                1.55x
    0.50             50%                1.73x
------------------------------------------------------------

Interpretation: Higher beta → workers care more about rents
                 → need higher wage compensation to move to expensive city


## Key Insights

1. **Spatial equilibrium**: Workers must get equal utility across locations, or they'll migrate
2. **Compensating differentials**: High rents must be offset by high wages or amenities
3. **Beta matters**: The housing expenditure share determines how sensitive workers are to rent differences
4. **Amenities are valued**: Cities with better amenities can have lower wages for given rents

This utility framework is the foundation for understanding:

- Why people live in expensive cities
- How housing costs affect migration patterns
- The welfare effects of housing supply constraints


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()